# 01 — Exploratory Data Analysis: ETTh1

This notebook explores the ETTh1 dataset to understand the target variable `OT` (oil temperature),
identify seasonal patterns, and justify the lag features (1 h, 24 h, 168 h) used in preprocessing.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

from src.data.download import download_data

# Download data if not present
download_data()

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/ETTh1.csv', parse_dates=['date'], index_col='date')
df = df.sort_index()
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
df.head()

## 2. Data Summary

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('NaN counts per column:')
print(df.isna().sum())

## 3. OT Time Series Plot

In [ ]:
fig = px.line(
    df,
    y='OT',
    title='Oil Temperature (OT) — Full Series',
    labels={'OT': 'Oil Temperature (°C)', 'date': 'Timestamp'},
)
fig.update_layout(hovermode='x unified')
fig.show()

## 4. ACF / PACF — Justifying Lags 1, 24, 168

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df['OT'].dropna(), lags=200, ax=axes[0], title='ACF — OT (200 lags)')
plot_pacf(df['OT'].dropna(), lags=200, ax=axes[1], title='PACF — OT (200 lags)')
for ax in axes:
    for lag in [1, 24, 168]:
        ax.axvline(lag, color='red', linestyle='--', alpha=0.6, label=f'lag {lag}')
axes[0].legend(['lag 1', 'lag 24', 'lag 168'], loc='upper right')
plt.tight_layout()
plt.savefig('../data/processed/acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Peaks at lags 1 (previous hour), 24 (daily), and 168 (weekly) confirm the chosen lag structure.')

## 5. Correlation Heatmap

In [ ]:
corr = df.corr()
fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Feature Correlation Heatmap',
)
fig.show()

## 6. Seasonal Decomposition

In [ ]:
decomp = seasonal_decompose(df['OT'].dropna(), model='additive', period=24)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed')
decomp.trend.plot(ax=axes[1], title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonal (period=24 h)')
decomp.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()